# 3 · Validate the instrument

**CPU-only. Runs the simulator live — takes ~1 minute.**

Before trusting any number measured on a real model, the *measure* has to be
shown to (a) stay quiet when nothing is happening and (b) fire in the right place
when something is. That's a claim about the instrument, so it can be checked
without a language model at all.

**Six pass conditions were declared before this code was first run.** If you want
to know whether the null in notebook 1 means anything, this is the notebook that
tells you.

In [1]:
import json, sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "code"))
DATA = ROOT / "data"

sim = json.loads((DATA / "simulation.json").read_text())
print("Declared before running:\n")
for c in sim["checks"]:
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}]  {c['check']}")

Declared before running:

  [PASS]  null: real arm matches filler arm (|gap| < 0.05 nats)
  [PASS]  anchor_random: filler gap is positive AT the planted step
  [PASS]  anchor_random: filler gap is ~zero away from the planted step
  [PASS]  anchor_random: planted step in top-3 raw (>60% of traces)
  [PASS]  anchor_random: planted step in top-3 after residualising (>60%)
  [PASS]  anchor_fixed: residualising attenuates the real anchor monotonically as bins get finer


## 3.1 The null world — is the estimator positional by construction?

A world where **no sentence causes anything**: the belief state drifts and
concentrates with depth, but nothing any sentence does matters.

If importance came out positional here, it would be positional *by arithmetic*,
and nothing measured on a real model could be interpreted either way.

In [2]:
null = next(r for r in sim["results"] if r["world"] == "null")
print("mean KL by position sextile (null world — nothing is happening):")
for k, v in enumerate(null["mean_kl_by_position_sextile"]):
    print(f"  sextile {k+1}: {v:.4f}  {'#' * int(v * 300)}")
print(f"\nposition-only rho = {null['position_only_spearman']:+.3f}")
print(f"real arm vs filler arm gap = {null['mean_real_minus_filler_all_positions']:+.4f}")
print("\nFlat. The estimator is NOT intrinsically positional — which is exactly")
print("what makes the real-data null in notebook 1 interpretable.")

mean KL by position sextile (null world — nothing is happening):
  sextile 1: 0.1546  ##############################################
  sextile 2: 0.1394  #########################################
  sextile 3: 0.1348  ########################################
  sextile 4: 0.1332  #######################################
  sextile 5: 0.1100  #################################
  sextile 6: 0.1167  ###################################

position-only rho = +0.141
real arm vs filler arm gap = +0.0166

Flat. The estimator is NOT intrinsically positional — which is exactly
what makes the real-data null in notebook 1 interpretable.


## 3.2 The anchor worlds — can it find a real effect?

A measure that never detects a planted anchor is useless regardless of how quiet
it is.

In [3]:
for name in ["anchor_random", "anchor_fixed"]:
    r = next(x for x in sim["results"] if x["world"] == name)
    print(f"--- {name} ---")
    print(f"  filler gap AT the planted step   : {r['planted_real_minus_filler']:+.3f}")
    print(f"  filler gap away from it          : {r['offtarget_real_minus_filler']:+.3f}")
    print(f"  planted step in top-3, raw       : {r['frac_top3_raw']:.0%}")
    print(f"  ... after residualising          : {r['frac_top3_residual']:.0%}\n")

--- anchor_random ---
  filler gap AT the planted step   : +0.521
  filler gap away from it          : -0.001
  planted step in top-3, raw       : 97%
  ... after residualising          : 100%

--- anchor_fixed ---
  filler gap AT the planted step   : +0.365
  filler gap away from it          : +0.005
  planted step in top-3, raw       : 81%
  ... after residualising          : 78%



The filler gap is large **at** the planted step and ~zero **away** from it.
The control localises a real effect precisely.

## 3.3 The prediction I got wrong here — and why it changed the design

I expected that residualising against position would **destroy** a real anchor
sitting at a fixed position. It doesn't; it *attenuates* it, and the attenuation
grows with the smoother's resolution.

In [4]:
fixed = next(r for r in sim["results"] if r["world"] == "anchor_fixed")
print(f"{'bins':>5} {'raw':>8} {'residual':>10} {'top-3':>8}")
for nb_, v in fixed["bin_sweep"].items():
    print(f"{nb_:>5} {v['mean_planted_raw']:8.3f} {v['mean_planted_residual']:10.3f} "
          f"{v['frac_top3_residual']:8.0%}")
print("\nSo the correlational control has a resolution knob that can delete real")
print("signal from anchors that genuinely cluster positionally. This is why the")
print("filler and paraphrase arms carry the conclusions, not the residualisation.")
print("All residuals in the report use 12 bins.")

 bins      raw   residual    top-3
    6    0.552      0.374      78%
   12    0.552      0.352      78%
   30    0.552      0.236      56%
   60    0.552      0.118      47%

So the correlational control has a resolution knob that can delete real
signal from anchors that genuinely cluster positionally. This is why the
filler and paraphrase arms carry the conclusions, not the residualisation.
All residuals in the report use 12 bins.


## 3.4 Re-run it yourself

The above is the saved output. Regenerate from scratch — about a minute on
CPU.

In [5]:
# Uncomment to re-run the full simulation (~1 min, CPU only):
# import subprocess
# print(subprocess.run([sys.executable, str(ROOT / "code" / "06_simulate.py"),
#                       "--traces", "32"], capture_output=True, text=True).stdout)

from anchors.simulate import SimConfig, simulate_trace
from anchors.importance import prefix_stats, sentence_importances

cfg = SimConfig(n_answers=6, n_sentences=30, n_rollouts=48, anchor_step=12)
sim_one = simulate_trace(cfg, seed=1)
stats_ = {i: prefix_stats(i, a, sim_one["gold"]) for i, a in sim_one["main"].items()}
imps = sentence_importances(stats_, cfg.n_sentences, sim_one["gold"], seed=1)

order = sorted(imps, key=lambda x: -x.kl_resampling)[:5]
print(f"planted anchor at step {cfg.anchor_step}. Top 5 by measured importance:")
for r in order:
    mark = "  <-- the planted anchor" if r.index == cfg.anchor_step else ""
    print(f"  step {r.index:2d}  KL = {r.kl_resampling:.3f}  "
          f"(floor {r.kl_null:.3f}){mark}")

planted anchor at step 12. Top 5 by measured importance:
  step 12  KL = 0.509  (floor 0.085)  <-- the planted anchor
  step  9  KL = 0.246  (floor 0.090)
  step  5  KL = 0.236  (floor 0.097)
  step 10  KL = 0.185  (floor 0.088)
  step  8  KL = 0.136  (floor 0.087)
